# 3. Coding attention mechanism

In [12]:
# necessary imports from previous notebook
# %run ch-2.ipynb

In [13]:
# inputs = input_embeddings

In [11]:
# inputs

In [14]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

## 3.3 Self attention without trainable weights

In [23]:
# was for educational purposes --> can be skipped

## 3.4 Self attention with trainable weights

In [15]:
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

### Building a custom class for SelfAttention

In [16]:
import torch.nn as nn

In [17]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query 
        values = x @ self.W_value
        attn_scores = queries @ keys.T #omega calculation
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim = -1)
        context_vec = attn_weights @ values
        return context_vec

In [18]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


### Modifying the SelfAttention using Pytorch's Linear layers

In [21]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        values = self.W_value(x)
        queries = self.W_query(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim = -1
        )
        context_vec = attn_weights @ values
        return context_vec

In [22]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Applying a causal attention mask

In [24]:
""" The current token should'nt have the understanding/context of the
future token so we mask them during the pretraining """

" The current token should'nt have the understanding/context of the\nfuture token so we mask them during the pretraining "

## Masking additionally - optional -- using dropout

In [27]:
batch = torch.stack((inputs, inputs), dim = 0)
print(batch.shape)

torch.Size([2, 6, 3])


In [34]:
class CausalAttention(nn.Module):
    def __init__(self, d_in: int, d_out: int, context_length: int, dropout: float, qkv_bias = False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        #adding a dropout layer by passing the dropout rate as the parameter
        self.dropout = nn.Dropout(dropout)
        #adding a mask by making the future value as -inf instead of 0; register buffer is used to make sure it can be sent to devices like gpu or other if required -> read the code in offical book for more information
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal = 1
        ))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        values = self.W_value(x)
        queries = self.W_query(x)
        #used .transpose(1, 2) instead of T to make sure we are transposing wrt context_length and d_in and keeping the batch_size at dim = 0 fixed
        attn_scores = queries @ keys.transpose(1, 2)
        #trailing _ means an inplace operation in Pytorch
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim = -1
        )
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

In [35]:
torch.manual_seed(123)
context_length = batch.shape[1]

In [36]:
ca = CausalAttention(d_in, d_out, context_length, 0.0)

In [37]:
context_vecs = ca(batch)

In [38]:
print(f"context_vecs.shape: {context_vecs.shape}")

context_vecs.shape: torch.Size([2, 6, 2])


In [39]:
context_vecs

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

## Stacking multiple single head attention heads

In [40]:
"""
It's quite easy actually, the hard part is already done by writing code for Single head -> Multi head is just concatenation of multiple heads
"""

"\nIt's quite easy actually, the hard part is already done by writing code for Single head -> Multi head is just concatenation of multiple heads\n"

In [47]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias = False):
        super().__init__()
        #this is a list only, but using standard Pytorch list for extra functionalities like register buffer and all
        #sequential bottle-neck --> no parallelization
        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)
        ])

    def forward(self, x):
        #just concatenating the contex_vec --> earlier the final dimension for each input's context vec was 2,,not it'll be 2 * num_heads
        return torch.cat([head(x) for head in self.heads], dim = -1)

In [48]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2

In [49]:
mha = MultiHeadAttentionWrapper(d_in ,d_out, context_length, 0.0, num_heads = 4)

In [50]:
context_vecs = mha(batch)

In [51]:
context_vecs.shape

torch.Size([2, 6, 8])

In [52]:
context_vecs

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063,  0.4566,  0.2729, -0.5684,
           0.5063],
         [-0.5874,  0.0058,  0.5891,  0.3257,  0.5792,  0.3011, -0.5388,
           0.6447],
         [-0.6300, -0.0632,  0.6202,  0.3860,  0.6249,  0.3102, -0.5242,
           0.6954],
         [-0.5675, -0.0843,  0.5478,  0.3589,  0.5691,  0.2785, -0.4578,
           0.6471],
         [-0.5526, -0.0981,  0.5321,  0.3428,  0.5543,  0.2520, -0.4006,
           0.5921],
         [-0.5299, -0.1081,  0.5077,  0.3493,  0.5337,  0.2499, -0.3997,
           0.5971]],

        [[-0.4519,  0.2216,  0.4772,  0.1063,  0.4566,  0.2729, -0.5684,
           0.5063],
         [-0.5874,  0.0058,  0.5891,  0.3257,  0.5792,  0.3011, -0.5388,
           0.6447],
         [-0.6300, -0.0632,  0.6202,  0.3860,  0.6249,  0.3102, -0.5242,
           0.6954],
         [-0.5675, -0.0843,  0.5478,  0.3589,  0.5691,  0.2785, -0.4578,
           0.6471],
         [-0.5526, -0.0981,  0.5321,  0.3428,  0.5543,  0.2520, -0.4

## Implementing multi-head attention with weight splits - single matrix will be used for all the heads

In [94]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias = False):
        super().__init__()
        assert(d_out % num_heads == 0), "d_out must be divisible by number of heads"

        #d_out is larger than before now = num_heads * old d_out (contex_vec length wrt one head)
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = self.d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        
        #linear layer to combine head outputs
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), 
                       diagonal = 1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        #breaking the keys, values and queries matrix into new dimensions -- granular view
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        #transposing from shapes (b, num_tokens, num_heads, head_dim) --> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        queries = queries.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim = -1
        )
        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ values).transpose(1, 2)

        #can use reshape directly without using contigouous operation --> good reasoning behind it -->read by making time!
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

        context_vec = self.out_proj(context_vec)
        return context_vec

In [95]:
torch.manual_seed(123)

In [112]:
batch_size, context_length, d_in = batch.shape

In [113]:
batch.shape

torch.Size([2, 6, 3])

In [137]:
d_out = 8

In [138]:
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads = 4)

In [139]:
context_vecs = mha(batch)

In [140]:
context_vecs

tensor([[[ 0.3484, -0.1888,  0.1460,  0.3430, -0.0985,  0.6994,  0.4255,
           0.1292],
         [ 0.4336, -0.1521,  0.1317,  0.3900, -0.1395,  0.7322,  0.4623,
          -0.0015],
         [ 0.4598, -0.1411,  0.1271,  0.4030, -0.1510,  0.7408,  0.4717,
          -0.0408],
         [ 0.4508, -0.1347,  0.1120,  0.4074, -0.1589,  0.6958,  0.4546,
          -0.0278],
         [ 0.4320, -0.1549,  0.1049,  0.3824, -0.1415,  0.6599,  0.4283,
           0.0195],
         [ 0.4407, -0.1427,  0.0983,  0.3980, -0.1561,  0.6573,  0.4363,
          -0.0016]],

        [[ 0.3484, -0.1888,  0.1460,  0.3430, -0.0985,  0.6994,  0.4255,
           0.1292],
         [ 0.4336, -0.1521,  0.1317,  0.3900, -0.1395,  0.7322,  0.4623,
          -0.0015],
         [ 0.4598, -0.1411,  0.1271,  0.4030, -0.1510,  0.7408,  0.4717,
          -0.0408],
         [ 0.4508, -0.1347,  0.1120,  0.4074, -0.1589,  0.6958,  0.4546,
          -0.0278],
         [ 0.4320, -0.1549,  0.1049,  0.3824, -0.1415,  0.6599,  0.4

In [141]:

context_vecs.shape

torch.Size([2, 6, 8])

In [142]:
"""
This was a great way to learn the things --> needs to build next 4 chapters in the same rhythm!
"""

'\nThis was a great way to learn the things --> needs to build next 4 chapters in the same rhythm!\n'